# Dataset Statistics and Split Verification
This notebook visualizes the overall dataset distributions and validates our leakage-free split.

In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cv2

base_dir = '..'
stats_path = os.path.join(base_dir, 'outputs', 'full_dataset_statistics.json')
train_split = os.path.join(base_dir, 'outputs', 'train_split.csv')
val_split = os.path.join(base_dir, 'outputs', 'val_split.csv')
json_path = os.path.join(base_dir, 'MAGFiLO_1.0_Kaggle_2026', 'train', 'MAGFiLO_1.0_Annotations_kaggle2026_train.json')

with open(json_path, 'r') as f:
    data = json.load(f)

In [ ]:
# Load splits
train_df = pd.read_csv(train_split)
val_df = pd.read_csv(val_split)

print(f"Train images: {len(train_df)}")
print(f"Val images: {len(val_df)}")

In [ ]:
# Calculate areas and counts
anns_by_img = {}
for ann in data['annotations']:
    img_id = ann['image_id']
    if img_id not in anns_by_img:
        anns_by_img[img_id] = []
    anns_by_img[img_id].append(ann)

def get_stats(df):
    areas = []
    counts = []
    for img_id in df['image_id']:
        anns = anns_by_img.get(img_id, [])
        counts.append(len(anns))
        for ann in anns:
            a = 0
            for seg in ann.get('segmentation', []):
                pts = np.array(seg, np.float32).reshape((-1, 1, 2))
                a += cv2.contourArea(pts)
            if a > 0: areas.append(a)
    return areas, counts

train_areas, train_counts = get_stats(train_df)
val_areas, val_counts = get_stats(val_df)

print(f"Train filaments: {sum(train_counts)}")
print(f"Val filaments: {sum(val_counts)}")

In [ ]:
# 1. Filament count distribution
plt.figure(figsize=(12, 5))
plt.hist([train_counts, val_counts], bins=20, label=['Train', 'Val'], density=True, color=['#1f77b4', '#ff7f0e'])
plt.legend()
plt.title('Filaments per Image Distribution (Train vs Val)')
plt.xlabel('Number of Filaments')
plt.ylabel('Density')
plt.show()

In [ ]:
# 2. Filament area distribution
plt.figure(figsize=(12, 5))
plt.hist([np.log1p(train_areas), np.log1p(val_areas)], bins=30, label=['Train', 'Val'], density=True, color=['#1f77b4', '#ff7f0e'])
plt.legend()
plt.title('Filament Area Distribution (Log Scale, Train vs Val)')
plt.xlabel('Log(Area + 1)')
plt.ylabel('Density')
plt.show()

In [ ]:
# 3. Display examples from Train and Val
def show_example(img_id, title):
    img_info = next(img for img in data['images'] if img['id'] == img_id)
    img_path = os.path.join(base_dir, 'MAGFiLO_1.0_Kaggle_2026', 'train', 'train_images', img_info['file_name'])
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # overlay masks
    overlay = img_rgb.copy()
    anns = anns_by_img.get(img_id, [])
    for ann in anns:
        for seg in ann.get('segmentation', []):
            pts = np.array(seg, np.int32).reshape((-1, 1, 2))
            cv2.fillPoly(overlay, [pts], (255, 0, 0))
    
    cv2.addWeighted(overlay, 0.5, img_rgb, 0.5, 0, img_rgb)
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis('off')

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
show_example(train_df['image_id'].iloc[0], 'Train Example')
plt.subplot(1, 2, 2)
show_example(val_df['image_id'].iloc[0], 'Val Example')
plt.show()